<a href="https://colab.research.google.com/github/alijon070/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('Menu Detector')

Menu Detector


In [12]:
# -----------------------
# Import Libraries
# -----------------------

from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.transforms  import transforms
from torchvision.models import mobilenet_v2

from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
import os
import numpy as np



In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)



Dataset_path: /content/drive/MyDrive/food101_dataset


In [5]:
CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping | class consolidation
    'cheesecake': 'dessert',     # label grouping | class consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASSES_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)
print('Classes:', CLASSES)
print('Classes to Index:', CLASSES_TO_IDX)
print('Number of Classes:', NUM_CLASSES)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Classes: ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
Classes to Index: {'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}
Number of Classes: 5


In [6]:
# -----------------------------
# Custom Dataset Class
# -----------------------------

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.images)

  def __getitem__(self, index):
    img_path = self.images[index]
    print('image_path', img_path)
    label = self.labels[index]
    print('label', label)
    try:
      image = Image.open(img_path).convert('RGB')
    except (UnidentifiedImageError, OSError):
      print(f"Skipping broken image: {img_path}")
      return self.__getitem__((index + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label

In [7]:
# -----------------------------
# Gather and Split Data
# -----------------------------
all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class)
  print('class_path:',  class_path)
  if not os.path.exists(class_path):
    print(f"Warning: {class_path}")
    continue
  for img in os.listdir(class_path):
    if img.endswith(('.jpg', '.jpeg', '.png')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASSES_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
test_data = all_images[split:]

train_images, train_labels = zip(*train_data)
test_images, test_labels = zip(*test_data)

print('Total images gathered:', len(all_images))

dataset = FoodDataset(train_images, train_labels)
print('Train Dataset Length:', len(dataset))
img, lbl = dataset[0]
print('Sample shape and label:', img.size, lbl)

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
Total images gathered: 4216
Train Dataset Length: 3372
image_path /content/drive/MyDrive/food101_dataset/cheesecake/3234179.jpg
label 2
Sample shape and label: (512, 512) 2


In [8]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
test_dataset = FoodDataset(test_images, test_labels, transform=transform)

In [9]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [10]:
# pretrained model
model = mobilenet_v2(weights="IMAGENET1K_V1") # pretrained model | lightweight
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning | backbone | model layer freeze
# model.eval()

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 162MB/s]


In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)
model = model.to(device)

device cuda


In [14]:
criterion = nn.CrossEntropyLoss() # Loss Function
optimizer = optim.Adam(model.parameters(), lr=0.001) # Optimizer
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick